Import Libraries

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

Load Dataset

In [2]:
df = pd.read_csv((r"C:\Users\Rumeth\Downloads\fertilizer_recommendation.csv"))
df.head()  # displays the first 5 rows of the dataset

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Nitrogen_Level,Phosphorus_Level,Potassium_Level,Temperature,Humidity,Rainfall,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Previous_Crop,Region,Fertilizer_Used_Last_Season,Yield_Last_Season,Recommended_Fertilizer
0,Clay,6.07,34.98,0.32,1.87,61,44,84,19.84,83.31,1693.22,Cotton,Harvest,Kharif,Canal,Wheat,South,297.15,1.19,MOP
1,Silt,6.39,47.34,0.28,0.21,59,56,18,24.40,46.27,1030.21,Maize,Vegetative,Kharif,Sprinkler,Potato,Central,77.17,4.40,Urea
2,Sandy,7.92,38.13,0.99,1.88,43,21,119,24.82,71.86,1166.16,Cotton,Flowering,Kharif,Rainfed,Tomato,South,128.93,7.21,Urea
3,Clay,5.86,14.17,1.46,0.36,88,46,34,27.87,53.23,2881.83,Wheat,Flowering,Zaid,Sprinkler,Potato,West,233.96,1.85,MOP
4,Clay,7.98,19.28,0.85,2.16,104,53,98,24.17,51.87,714.84,Potato,Sowing,Kharif,Rainfed,Maize,East,214.39,7.36,Zinc Sulphate


Features and Target Seperation

In [3]:
X = df.drop(columns=["Recommended_Fertilizer", "Season", "Region"])  # features for prediction
Y = df["Recommended_Fertilizer"]  # target variable for prediction

print("Features shape:", X.shape)  # prints the shape of the features
print("Target shape:", Y.shape)  # prints the shape of the target variable

Features shape: (10000, 17)
Target shape: (10000,)


Check Unique Classes

In [4]:
print("Number of unique fertilizers:", Y.nunique())  # prints the number of unique fertilizers in the target variable

print("\nUnique fertilizers and their counts:")  # prints the unique fertilizers and their counts
print(Y.value_counts())  # prints the counts of each unique fertilizer in the target variable

Number of unique fertilizers: 7

Unique fertilizers and their counts:
Recommended_Fertilizer
Urea             3101
DAP              2920
MOP              1408
Compost           996
Zinc Sulphate     752
NPK               641
SSP               182
Name: count, dtype: int64


Dataset Split into Training and Testing

In [5]:
X_train, X_test, Y_train, Y_test = train_test_split(  
    X,      # Input features
    Y,      # Target variable
    test_size=0.2,  # Allocates 20% of the data for testing and 80% for training
    random_state=42,    # Ensures reproducibility of the split
    stratify=Y  # Maintains the class distribution in both sets
    )  

print("Training data:", X_train.shape) # Display the dimensions of the training dataset
print("Testing data:", X_test.shape) # Display the dimensions of the testing dataset

Training data: (8000, 17)
Testing data: (2000, 17)


Verify Class Distribution in Testing and Training Sets

In [6]:
print("Training target distribution:")
print(Y_train.value_counts(normalize=True).round(3))

print("\nTesting target distribution:")
print(Y_test.value_counts(normalize=True).round(3))

Training target distribution:
Recommended_Fertilizer
Urea             0.310
DAP              0.292
MOP              0.141
Compost          0.100
Zinc Sulphate    0.075
NPK              0.064
SSP              0.018
Name: proportion, dtype: float64

Testing target distribution:
Recommended_Fertilizer
Urea             0.310
DAP              0.292
MOP              0.141
Compost          0.100
Zinc Sulphate    0.075
NPK              0.064
SSP              0.018
Name: proportion, dtype: float64


Create Preprocessing Pipeline

In [8]:
# Seperating numerical and categorical features 
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["str"]
).columns.tolist()

Numeric Pipeline

In [11]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median") # handles missing values by replacing them with the median of the column
        ),
        (
            "scaler",
            StandardScaler() # standardizes features by removing the mean and scaling to unit variance
        )
    ]
)

Categorical Pipeline

In [12]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",  # fills missing values with a constant value
                fill_value="Unknown"  # fills missing values with the string "Unknown"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore" # ignores any unknown categories encountered during transformation
            )
        )
    ]
)

Combined Preprocessor

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numerical_features  # applies the numeric_transformer to the numerical features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features  # applies the categorical_transformer to the categorical features
        )
    ]
) 

Preprocessing the Training and Testing Data

In [15]:
X_train_preprocessed = preprocessor.fit_transform(X_train)  # fits the preprocessor to the training data and transforms it
X_test_preprocessed = preprocessor.transform(X_test)  # transforms the testing data using the fitted preprocessor

print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_preprocessed.shape)

print("\nOriginal testing shape:", X_test.shape)
print("Processed testing shape:", X_test_preprocessed.shape)

Original training shape: (8000, 17)
Processed training shape: (8000, 38)

Original testing shape: (2000, 17)
Processed testing shape: (2000, 38)
